# B3 — re-evaluating every follow-up checkpoint against an independent fine reference

**Why this notebook exists**: Professor Rabczuk's own review of the follow-up investigation report caught a real methodological gap -- every region-stress, H1 semi-norm, and break-even number in the follow-up study (items 1-4) was measured against the SAME 6,840-element mesh each checkpoint was trained on, not an independent finer reference. The submitted paper itself already established two real, independently-validated fine references for exactly this purpose:

- **950,400 elements** `(101,100,97)` -- the paper's own 13-point mesh-convergence ladder's finest row, confirmed converged against the second-finest (839,040-element) row to within 0.86%, and adopted in the paper as "the common reference used throughout the 3D results." Used here for **region Cauchy stress** and **H1 semi-norm**.
- **43,400 elements** `(41,36,32)` -- a separate, independently-established reference the paper uses for its own accuracy-matched displacement/energy/reaction analysis. Used here for the **break-even sweep's displacement comparison** specifically, so PI-vs-DD is finally apples-to-apples (the follow-up's own break-even numbers previously quoted "PI baseline 1.88%" -- the RAW, uncorrected number against the 6,840-element mesh, not the paper's own accuracy-matched 2.03%).

**The expensive part (re-solving held-out samples at each fine mesh) runs ONCE per reference**, not once per checkpoint -- a real, measured engineering fix (see `evaluate_B3_qois_finer.py`'s own module docstring) that keeps this notebook's GPU cost to roughly 10 fine-mesh FEM solves at each resolution, not 10x that for every checkpoint re-evaluated against it.

**Checkpoints re-evaluated** (against the 950,400-element reference, region stress + H1): PI baseline, item 1 attempts 1 and 5 (mixed-stress, both the implicit AND direct stress paths), items 2/3 attempts 1 and 2 (equilibrium-augmented), item 4's rel_l2/sobolev/hybrid losses. **Against the 43,400-element reference (displacement only)**: PI baseline, and the break-even sweep's N=50/200/400 data-driven checkpoints.

In [ ]:
# =====================================================================
#  CELL -- Re-evaluate every follow-up B3 checkpoint against the two
#  independently-established fine references (950,400 el for stress/H1,
#  43,400 el for the break-even displacement comparison), instead of
#  each checkpoint's own 6,840-element training mesh.
# =====================================================================
import os
os.environ['JAX_PLATFORMS'] = 'cpu'

import subprocess
import sys
import time

_started = time.time()


def run(cmd):
    print('$', ' '.join(str(c) for c in cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if p.returncode != 0:
        raise subprocess.CalledProcessError(p.returncode, cmd)


from google.colab import drive
if os.path.isdir('/content/drive/MyDrive'):
    print('Drive already mounted at /content/drive.')
else:
    drive.mount('/content/drive')

REPO = '/content/OMAR'
if not os.path.isdir(REPO):
    run(['git', 'clone', '-b', 'claude/claude-code-question-d307wp',
         'https://github.com/SUHIBAMRO/OMAR.git', REPO])
else:
    run(['git', '-C', REPO, 'fetch', 'origin', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'checkout', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'reset', '--hard', 'origin/claude/claude-code-question-d307wp'])

run([sys.executable, '-m', 'pip', 'install', '-q', 'pyvista<0.49'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-fem'])

WORK = f'{REPO}/Practical_Examples'
os.chdir(WORK)
sys.path.insert(0, WORK)

for _mod_name in list(sys.modules):
    if (_mod_name == 'omar_pfem' or _mod_name.startswith('omar_pfem.')
            or _mod_name == 'torchfem' or _mod_name.startswith('torchfem.')
            or _mod_name == 'pyvista' or _mod_name.startswith('pyvista.')):
        del sys.modules[_mod_name]

import torch
assert torch.cuda.is_available(), 'this cell needs a real GPU'
print('GPU:', torch.cuda.get_device_name(0))

import gc
for _attr in ('last_traceback', 'last_value', 'last_type'):
    if hasattr(sys, _attr):
        delattr(sys, _attr)
gc.collect()
torch.cuda.empty_cache()

from omar_pfem.evaluate_B3_qois_finer import (
    generate_fine_reference, evaluate_u_only_checkpoint_against_fine_reference,
    FINE_RESOLUTION_STRESS, FINE_RESOLUTION_DISP_ENERGY_REACTION,
)
from omar_pfem.evaluate_B3_mixed_stress_finer import evaluate_mixed_stress_checkpoint_against_fine_reference

device = torch.device('cuda')
dtype = torch.float64
R = '/content/drive/MyDrive/pfem_run'
HOLDOUT = f'{R}/b3_dataset_clean_holdout/dataset.h5'
N_SAMPLES = 10

print(f'\n{"=" * 90}')
print(f'=== Generating the 950,400-element ({FINE_RESOLUTION_STRESS}) fine reference, {N_SAMPLES} samples ===')
print('=== (region stress + H1 semi-norm reference -- the paper\'s own mesh-convergence choice) ===')
t0 = time.time()
fine_ref_stress = generate_fine_reference(HOLDOUT, FINE_RESOLUTION_STRESS, N_SAMPLES, device, dtype)
print(f'Fine reference (stress/H1) generated in {time.time() - t0:.1f}s.')

U_ONLY_CHECKPOINTS = [
    ('PI baseline', f'{R}/b3_training_normalized/checkpoint_50000.pt'),
    ('Items 2/3, attempt 1 (equilibrium, w=1.0)', f'{R}/b3_equilibrium_augmented/checkpoint_50000.pt'),
    ('Items 2/3, attempt 2 (equilibrium, w=0.12)', f'{R}/b3_equilibrium_augmented_v2/checkpoint_50000.pt'),
    ('Item 4, data-driven rel_l2 (= break-even N=200)', f'{R}/b3_data_driven_rel_l2/model_best.pt'),
    ('Item 4, data-driven sobolev', f'{R}/b3_data_driven_sobolev/model_best.pt'),
    ('Item 4, data-driven hybrid', f'{R}/b3_data_driven_hybrid/model_best.pt'),
]
MIXED_STRESS_CHECKPOINTS = [
    ('Item 1, attempt 1 (mixed-stress, scale=500)', f'{R}/b3_mixed_stress_training/checkpoint_50000.pt'),
    ('Item 1, attempt 5 (mixed-stress, local-refine)', f'{R}/b3_mixed_stress_local_refine/checkpoint_50000.pt'),
]

results_stress = {}
for name, ckpt in U_ONLY_CHECKPOINTS:
    print(f'\n{"=" * 90}')
    print(f'=== [stress/H1 @ 950,400 el] {name} ===')
    if not os.path.exists(ckpt):
        print(f'  SKIPPED -- checkpoint not found at {ckpt}')
        continue
    results_stress[name] = evaluate_u_only_checkpoint_against_fine_reference(ckpt, fine_ref_stress, device, dtype)

for name, ckpt in MIXED_STRESS_CHECKPOINTS:
    print(f'\n{"=" * 90}')
    print(f'=== [stress/H1 @ 950,400 el, implicit + direct] {name} ===')
    if not os.path.exists(ckpt):
        print(f'  SKIPPED -- checkpoint not found at {ckpt}')
        continue
    results_stress[name] = evaluate_mixed_stress_checkpoint_against_fine_reference(ckpt, fine_ref_stress, device, dtype)

print(f'\n{"=" * 90}')
print(f'=== Generating the 43,400-element ({FINE_RESOLUTION_DISP_ENERGY_REACTION}) fine reference, {N_SAMPLES} samples ===')
print('=== (displacement reference -- the paper\'s own accuracy-matched choice, for the break-even comparison) ===')
t0 = time.time()
fine_ref_disp = generate_fine_reference(HOLDOUT, FINE_RESOLUTION_DISP_ENERGY_REACTION, N_SAMPLES, device, dtype)
print(f'Fine reference (displacement) generated in {time.time() - t0:.1f}s.')

BREAKEVEN_CHECKPOINTS = [
    ('PI baseline', f'{R}/b3_training_normalized/checkpoint_50000.pt'),
    ('DD rel_l2 @ N=50', f'{R}/b3_data_driven_rel_l2_N50/model_best.pt'),
    ('DD rel_l2 @ N=200 (existing)', f'{R}/b3_data_driven_rel_l2/model_best.pt'),
    ('DD rel_l2 @ N=400', f'{R}/b3_data_driven_rel_l2_N400/model_best.pt'),
]
results_disp = {}
for name, ckpt in BREAKEVEN_CHECKPOINTS:
    print(f'\n{"=" * 90}')
    print(f'=== [displacement @ 43,400 el] {name} ===')
    if not os.path.exists(ckpt):
        print(f'  SKIPPED -- checkpoint not found at {ckpt}')
        continue
    results_disp[name] = evaluate_u_only_checkpoint_against_fine_reference(ckpt, fine_ref_disp, device, dtype)

print(f'\n{"=" * 90}')
print('=== SUMMARY: region stress + H1 semi-norm, vs. the 950,400-element reference ===')
for name, r in results_stress.items():
    s = r['summary']
    if 'mean_region_cauchy_field_rel_fine' in s:  # u-only path
        print(f'  {name}:')
        print(f'    region_cauchy_field_rel (vs. 950,400-el): mean={s["mean_region_cauchy_field_rel_fine"]:.4f}  '
              f'median={s["median_region_cauchy_field_rel_fine"]:.4f}')
        print(f'    H1_seminorm_rel (vs. 950,400-el):         mean={s["mean_H1_seminorm_rel_fine"]:.4f}  '
              f'median={s["median_H1_seminorm_rel_fine"]:.4f}')
    else:  # mixed-stress path
        print(f'  {name}:')
        print(f'    region_cauchy_field_rel IMPLICIT (vs. 950,400-el): mean={s["mean_region_cauchy_field_rel_implicit_fine"]:.4f}')
        print(f'    region_cauchy_field_rel DIRECT   (vs. 950,400-el): mean={s["mean_region_cauchy_field_rel_direct_fine"]:.4f}')
        print(f'    H1_seminorm_rel (vs. 950,400-el):                  mean={s["mean_H1_seminorm_rel_fine"]:.4f}')

print(f'\n{"=" * 90}')
print('=== SUMMARY: break-even displacement, vs. the 43,400-element reference (accuracy-matched) ===')
for name, r in results_disp.items():
    s = r['summary']
    print(f'  {name}: disp_rel_l2_combined (vs. 43,400-el) = {s["mean_disp_rel_l2_fine_combined"]:.4f}')

try:
    from omar_pfem.run_manifest import write_manifest
    write_manifest(f'{R}/b3_fine_reference_reevaluation', kind='b3_fine_reference_reevaluation',
                    args={'fine_resolution_stress': list(FINE_RESOLUTION_STRESS),
                          'fine_resolution_disp': list(FINE_RESOLUTION_DISP_ENERGY_REACTION),
                          'n_samples': N_SAMPLES,
                          'checkpoints_stress': [n for n, _ in U_ONLY_CHECKPOINTS + MIXED_STRESS_CHECKPOINTS],
                          'checkpoints_disp': [n for n, _ in BREAKEVEN_CHECKPOINTS]},
                    started_at=_started,
                    results={'elapsed_total_s': time.time() - _started},
                    outputs=[],
                    notes="Re-evaluates every follow-up-investigation B3 checkpoint against the "
                          "paper's own two independently-established fine references (950,400 el "
                          "for stress/H1, 43,400 el for displacement), closing the gap Professor "
                          "Rabczuk's review identified: none of the follow-up's own numbers had "
                          "previously been checked against anything but the 6,840-element "
                          "training mesh.")
except Exception as e:
    print(f'[manifest] not recorded: {e}')

print(f'\n{"=" * 90}')
print(f'Total wall-clock: {time.time() - _started:.1f}s')
print('\nDone. Send this cell\'s full printed output back -- both summary sections above.')
